# Unit 3 Assignment: Building a Production Advanced RAG System

**Topic:** Advanced RAG — Retrieval Enhancement, Re-Ranking, and Query Expansion  
**Tools:** Python, HuggingFace, Groq API, Google Gemini API, rank-bm25, sentence-transformers


In [1]:
# Install all required packages
!pip install rank-bm25 sentence-transformers groq google-generativeai numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.8 MB/s eta 0:00:00


## Step 1 — Imports & API Keys

In [2]:
import os
import numpy as np
from typing import List, Dict
import getpass

# BM25
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from groq import Groq
import google.generativeai as genai

GROQ_API_KEY   = getpass.getpass("Enter your GROQ API key: ")
GEMINI_API_KEY = getpass.getpass("Enter your GEMINI API key: ")

# Configure clients
groq_client = Groq(api_key=GROQ_API_KEY)
genai.configure(api_key=GEMINI_API_KEY)

print("Imports and API clients configured successfully.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Enter your GROQ API key: ··········
Enter your GEMINI API key: ··········
Imports and API clients configured successfully.


---
## Part 1 — Document Corpus Setup

I create a corpus of 12 documents on AI/ML topics:  
- At least 3 on related but distinct sub-topics (neural network training)  
- At least 1 with technical proper nouns that BM25 excels at

In [3]:
corpus = [
    # --- Neural Network Training (3 docs from different angles) ---
    # Doc 0: Gradient descent perspective
    "Neural network training uses gradient descent to minimize a loss function by iteratively "
    "adjusting weights in the direction of the negative gradient, making small updates each step.",

    # Doc 1: Backpropagation perspective
    "Backpropagation computes gradients layer-by-layer using the chain rule of calculus, "
    "allowing credit to be assigned to each weight based on its contribution to the total error.",

    # Doc 2: Regularisation / overfitting perspective
    "To prevent overfitting during neural network training, techniques such as dropout, "
    "L2 regularization, and early stopping are applied to improve generalization on unseen data.",

    # --- Transformers & Attention ---
    # Doc 3
    "The Transformer architecture relies on self-attention mechanisms to weigh the relevance of "
    "every token against every other token in a sequence, enabling rich contextual representations.",

    # Doc 4
    "Attention in neural networks assigns a score to each word pair in a sentence; the softmax "
    "of these scores determines how much focus each token receives when building its representation.",

    # Doc 5
    "BERT (Bidirectional Encoder Representations from Transformers) pre-trains on masked language "
    "modelling and next-sentence prediction, learning deep bidirectional context for downstream NLP tasks.",

    # --- Optimization ---
    # Doc 6
    "Adam optimizer combines momentum and adaptive learning rates, maintaining per-parameter "
    "running averages of gradients and squared gradients to adjust step sizes automatically.",

    # Doc 7
    "Learning rate scheduling techniques such as cosine annealing and warm restarts help "
    "the optimizer escape local minima and converge to flatter, more generalizable solutions.",

    # --- Embeddings & Representations ---
    # Doc 8
    "Word embeddings such as Word2Vec and GloVe map tokens to dense vectors in a continuous "
    "semantic space where similar words are geometrically close to each other.",

    # Doc 9 — Technical jargon + proper nouns (BM25 advantage)
    "The BLEU score (Bilingual Evaluation Understudy) measures n-gram overlap between a "
    "machine translation hypothesis and reference translations; it is computed as a geometric "
    "mean of precision scores multiplied by a brevity penalty to penalize short outputs.",

    # --- Convolutional Neural Networks ---
    # Doc 10
    "Convolutional Neural Networks (CNNs) use learnable filters that slide over input images "
    "to detect local features such as edges and textures, building hierarchical representations "
    "through successive pooling and convolution layers.",

    # --- Reinforcement Learning ---
    # Doc 11
    "Reinforcement learning trains an agent to maximize cumulative reward by interacting with "
    "an environment; policy gradient methods like PPO (Proximal Policy Optimization) directly "
    "optimize the expected return using sampled trajectories.",
]

# Verify corpus size
print(f"Corpus size: {len(corpus)} documents")
print("\nDocument listing:")
for i, doc in enumerate(corpus):
    print(f"  [{i:02d}] {doc[:80]}...")

Corpus size: 12 documents

Document listing:
  [00] Neural network training uses gradient descent to minimize a loss function by ite...
  [01] Backpropagation computes gradients layer-by-layer using the chain rule of calcul...
  [02] To prevent overfitting during neural network training, techniques such as dropou...
  [03] The Transformer architecture relies on self-attention mechanisms to weigh the re...
  [04] Attention in neural networks assigns a score to each word pair in a sentence; th...
  [05] BERT (Bidirectional Encoder Representations from Transformers) pre-trains on mas...
  [06] Adam optimizer combines momentum and adaptive learning rates, maintaining per-pa...
  [07] Learning rate scheduling techniques such as cosine annealing and warm restarts h...
  [08] Word embeddings such as Word2Vec and GloVe map tokens to dense vectors in a cont...
  [09] The BLEU score (Bilingual Evaluation Understudy) measures n-gram overlap between...
  [10] Convolutional Neural Networks (CNNs) u

---
## Part 2 — Hybrid Retriever (BM25 + SBERT + RRF)

The HybridRetriever class:
1. Builds a BM25 index over the tokenised corpus
2. Builds an SBERT dense index (cosine similarity)
3. Fuses both ranked lists with Reciprocal Rank Fusion (RRF)



In [4]:
class HybridRetriever:

    def __init__(self, corpus: List[str], k: int = 60):

        self.corpus = corpus
        self.k = k

        # BM25 setup
        tokenised_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenised_corpus)
        print("BM25 index built.")

        # SBERT setup
        self.sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.corpus_embeddings = self.sbert_model.encode(
            corpus, convert_to_numpy=True, show_progress_bar=True
        )
        print("SBERT index built.")

    def _bm25_ranked(self, query: str) -> List[int]:
        tokenised_query = query.lower().split()
        scores = self.bm25.get_scores(tokenised_query)
        ranked = np.argsort(scores)[::-1].tolist()
        return ranked

    def _sbert_ranked(self, query: str) -> List[int]:
        query_embedding = self.sbert_model.encode([query], convert_to_numpy=True)

        corpus_norms = np.linalg.norm(self.corpus_embeddings, axis=1, keepdims=True)
        query_norm   = np.linalg.norm(query_embedding)
        similarities = (self.corpus_embeddings @ query_embedding.T).flatten() / (
            corpus_norms.flatten() * query_norm + 1e-9
        )
        ranked = np.argsort(similarities)[::-1].tolist()
        return ranked

    def _rrf_score(self, rank: int) -> float:
        """
        Computes the RRF contribution of a single rank position.
        RRF(d) = 1 / (k + rank)   where rank is 1-indexed.
        """
        return 1.0 / (self.k + rank)

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        bm25_ranking  = self._bm25_ranked(query)
        sbert_ranking = self._sbert_ranked(query)

        bm25_rank_map  = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranking)}
        sbert_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranking)}

        rrf_scores: Dict[int, float] = {}
        for doc_id in range(len(self.corpus)):
            rrf_scores[doc_id] = (
                self._rrf_score(bm25_rank_map[doc_id]) +
                self._rrf_score(sbert_rank_map[doc_id])
            )

        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for doc_id, rrf_score in sorted_docs[:top_k]:
            results.append({
                "doc_id"     : doc_id,
                "rrf_score"  : round(rrf_score, 6),
                "bm25_rank"  : bm25_rank_map[doc_id],
                "sbert_rank" : sbert_rank_map[doc_id],
                "text"       : self.corpus[doc_id],
            })

        return results

In [5]:
retriever = HybridRetriever(corpus=corpus, k=60)
print("\nSmoke-test: 'how do transformers encode meaning?'")
test_results = retriever.retrieve("how do transformers encode meaning?", top_k=5)
for r in test_results:
    print(f"  doc_id={r['doc_id']}  rrf={r['rrf_score']}  "
          f"bm25_rank={r['bm25_rank']}  sbert_rank={r['sbert_rank']}")
    print(f"    → {r['text'][:80]}...")

BM25 index built.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT index built.

Smoke-test: 'how do transformers encode meaning?'
  doc_id=4  rrf=0.032018  bm25_rank=1  sbert_rank=4
    → Attention in neural networks assigns a score to each word pair in a sentence; th...
  doc_id=10  rrf=0.031498  bm25_rank=4  sbert_rank=3
    → Convolutional Neural Networks (CNNs) use learnable filters that slide over input...
  doc_id=9  rrf=0.031258  bm25_rank=3  sbert_rank=5
    → The BLEU score (Bilingual Evaluation Understudy) measures n-gram overlap between...
  doc_id=3  rrf=0.030886  bm25_rank=9  sbert_rank=1
    → The Transformer architecture relies on self-attention mechanisms to weigh the re...
  doc_id=5  rrf=0.030835  bm25_rank=8  sbert_rank=2
    → BERT (Bidirectional Encoder Representations from Transformers) pre-trains on mas...


---
## Part 3 — Cross-Encoder Re-Ranker

  
Accepts the original user query (not the HyDE-expanded version).  
Returns top-k re-ranked documents with cross-encoder scores.  
Scores can be negative — higher (less negative) = more relevant.

In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder model loaded.")


def rerank(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    if not candidates:
        return []
    pairs = [(query, cand["text"]) for cand in candidates]

    scores = cross_encoder.predict(pairs)

    scored_candidates = []
    for cand, score in zip(candidates, scores):
        entry = dict(cand)
        entry["cross_encoder_score"] = float(score)
        scored_candidates.append(entry)

    scored_candidates.sort(key=lambda x: x["cross_encoder_score"], reverse=True)

    return scored_candidates[:top_k]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder model loaded.


In [7]:
sample_query = "how do transformers encode meaning?"
sample_candidates = retriever.retrieve(sample_query, top_k=5)
reranked = rerank(query=sample_query, candidates=sample_candidates, top_k=3)

print("Re-ranked results:")
for i, r in enumerate(reranked, 1):
    print(f"  {i}. CE_score={r['cross_encoder_score']:.4f}  doc_id={r['doc_id']}")
    print(f"     → {r['text'][:90]}...")

Re-ranked results:
  1. CE_score=1.9282  doc_id=5
     → BERT (Bidirectional Encoder Representations from Transformers) pre-trains on masked langua...
  2. CE_score=-8.4698  doc_id=3
     → The Transformer architecture relies on self-attention mechanisms to weigh the relevance of...
  3. CE_score=-11.1430  doc_id=9
     → The BLEU score (Bilingual Evaluation Understudy) measures n-gram overlap between a machine...


---
## Part 4 — Query Expansion (HyDE via Gemini)

In [12]:
def hyde_expand(user_query: str) -> str:
    prompt = (
        "You are an expert in AI and machine learning. "
        "Write a factual, concise answer (2-3 sentences) to the following question "
        "using precise technical vocabulary. "
        "Do NOT say 'I' or mention that this is hypothetical. "
        "Just write the answer directly as if it were a passage from a textbook.\n\n"
        f"Question: {user_query}"
    )

    gemini_model = genai.GenerativeModel("gemini-2.5-flash")
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0.0, max_output_tokens=150),
    )
    hypothetical_doc = response.text.strip()
    return hypothetical_doc

In [13]:
test_query = "how do transformers encode meaning?"
hyp_doc = hyde_expand(test_query)
print(f"Original query : {test_query}")
print(f"\nHyDE expansion :\n{hyp_doc}")

Original query : how do transformers encode meaning?

HyDE expansion :
Transformers encode meaning by generating



## Part 5 — End-to-End Advanced RAG Pipeline


In [29]:
def advanced_rag(user_query: str) -> str:
    print(f"\n{'='*60}")
    print(f"User query: {user_query}")

    # Step 1: HyDE Query Expansion
    print("\n[1/4] HyDE query expansion ...")
    expanded_query = hyde_expand(user_query)
    print(f"      Expanded: {expanded_query[:120]}...")

    # Step 2: Hybrid Retrieval on expanded query
    print("\n[2/4] Hybrid retrieval (BM25 + SBERT + RRF) ...")
    candidates = retriever.retrieve(expanded_query, top_k=5)
    print(f"      Retrieved {len(candidates)} candidates:")
    for c in candidates:
        print(f"        doc={c['doc_id']}  rrf={c['rrf_score']}  "
              f"bm25_rank={c['bm25_rank']}  sbert_rank={c['sbert_rank']}")

    #Step 3: Cross-Encoder Re-ranking on ORIGINAL query
    print("\n[3/4] Cross-encoder re-ranking (original query) ...")
    top_docs = rerank(query=user_query, candidates=candidates, top_k=3)
    print(f"      Top-3 after re-ranking:")
    for i, d in enumerate(top_docs, 1):
        print(f"        {i}. CE_score={d['cross_encoder_score']:.4f}  doc_id={d['doc_id']}")

    # Step 4: LLM Generation with Groq
    print("\n[4/4] Generating answer with Groq LLM ...")

    context = "\n\n".join(
        [f"[{i+1}] {doc['text']}" for i, doc in enumerate(top_docs)]
    )

    system_prompt = (
        "You are a helpful university AI/ML teaching assistant. "
        "Answer the student's question based ONLY on the provided context. "
        "Be clear, concise, and accurate. "
        "If the context does not contain enough information, say so."
    )

    user_prompt = (
        f"Context:\n{context}\n\n"
        f"Student Question: {user_query}\n\n"
        "Answer:"
    )

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=300,
    )

    answer = response.choices[0].message.content.strip()
    print(f"\nFinal Answer:\n{answer}")
    print('='*60)
    return answer

In [30]:
ans1 = advanced_rag("how do transformers encode meaning?")


User query: how do transformers encode meaning?

[1/4] HyDE query expansion ...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1878.20ms


      Expanded: Transformers encode meaning by generating...

[2/4] Hybrid retrieval (BM25 + SBERT + RRF) ...
      Retrieved 5 candidates:
        doc=9  rrf=0.031258  bm25_rank=3  sbert_rank=5
        doc=10  rrf=0.03125  bm25_rank=4  sbert_rank=4
        doc=5  rrf=0.031054  bm25_rank=7  sbert_rank=2
        doc=0  rrf=0.030886  bm25_rank=1  sbert_rank=9
        doc=3  rrf=0.030679  bm25_rank=10  sbert_rank=1

[3/4] Cross-encoder re-ranking (original query) ...
      Top-3 after re-ranking:
        1. CE_score=1.9282  doc_id=5
        2. CE_score=-8.4698  doc_id=3
        3. CE_score=-11.1430  doc_id=9

[4/4] Generating answer with Groq LLM ...

Final Answer:
According to the context, the Transformer architecture encodes meaning through self-attention mechanisms, which weigh the relevance of every token against every other token in a sequence, enabling rich contextual representations.


---
## Part 6 — Naïve RAG Baseline


In [19]:
def naive_rag(user_query: str) -> Dict:

    query_embedding = retriever.sbert_model.encode([user_query], convert_to_numpy=True)

    corpus_norms = np.linalg.norm(retriever.corpus_embeddings, axis=1, keepdims=True)
    query_norm   = np.linalg.norm(query_embedding)
    similarities = (retriever.corpus_embeddings @ query_embedding.T).flatten() / (
        corpus_norms.flatten() * query_norm + 1e-9
    )

    # Top-3 by cosine similarity
    top_indices = np.argsort(similarities)[::-1][:3]
    top_docs    = [corpus[i] for i in top_indices]

    context = "\n\n".join([f"[{i+1}] {doc}" for i, doc in enumerate(top_docs)])

    system_prompt = (
        "You are a helpful university AI/ML teaching assistant. "
        "Answer the student's question based ONLY on the provided context."
    )
    user_prompt = (
        f"Context:\n{context}\n\n"
        f"Student Question: {user_query}\n\n"
        "Answer:"
    )

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=300,
    )

    answer = response.choices[0].message.content.strip()

    return {
        "top_doc"      : corpus[top_indices[0]],
        "top_doc_index": int(top_indices[0]),
        "answer"       : answer,
    }

---
## Part 6 — Comparison Experiment

Running all 3 test queries through both pipelines.

In [20]:
def advanced_rag_top_doc(user_query: str) -> Dict:

    expanded_query = hyde_expand(user_query)
    candidates     = retriever.retrieve(expanded_query, top_k=5)
    top_docs       = rerank(query=user_query, candidates=candidates, top_k=3)

    context = "\n\n".join([f"[{i+1}] {doc['text']}" for i, doc in enumerate(top_docs)])

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful university AI/ML teaching assistant. Answer based ONLY on the context."},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {user_query}\n\nAnswer:"},
        ],
        temperature=0.2,
        max_tokens=300,
    )

    return {
        "top_doc"      : top_docs[0]["text"],
        "top_doc_index": top_docs[0]["doc_id"],
        "ce_score"     : top_docs[0]["cross_encoder_score"],
        "answer"       : response.choices[0].message.content.strip(),
    }

In [21]:
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "how does backpropagation assign credit to weights?",   # custom query
]

comparison_results = []

for query in test_queries:
    print(f"\n{'─'*60}")
    print(f"Query: {query}")

    # Naïve RAG
    print("  Running Naïve RAG...")
    naive_result = naive_rag(query)

    # Advanced RAG
    print("  Running Advanced RAG...")
    adv_result = advanced_rag_top_doc(query)

    are_different = naive_result["top_doc_index"] != adv_result["top_doc_index"]

    comparison_results.append({
        "query"             : query,
        "naive_top_doc_id"  : naive_result["top_doc_index"],
        "naive_top_doc"     : naive_result["top_doc"][:80] + "...",
        "naive_answer"      : naive_result["answer"],
        "adv_top_doc_id"    : adv_result["top_doc_index"],
        "adv_top_doc"       : adv_result["top_doc"][:80] + "...",
        "adv_answer"        : adv_result["answer"],
        "are_different"     : are_different,
    })

    print(f"  Naïve top doc  [doc {naive_result['top_doc_index']}]: {naive_result['top_doc'][:60]}...")
    print(f"  Advanced top doc [doc {adv_result['top_doc_index']}]: {adv_result['top_doc'][:60]}...")
    print(f"  Different? {'YES' if are_different else 'NO (same doc)'}")

print("\nAll queries processed.")


────────────────────────────────────────────────────────────
Query: how do transformers encode meaning?
  Running Naïve RAG...
  Running Advanced RAG...
  Naïve top doc  [doc 3]: The Transformer architecture relies on self-attention mechan...
  Advanced top doc [doc 5]: BERT (Bidirectional Encoder Representations from Transformer...
  Different? YES

────────────────────────────────────────────────────────────
Query: optimization techniques for training
  Running Naïve RAG...
  Running Advanced RAG...
  Naïve top doc  [doc 7]: Learning rate scheduling techniques such as cosine annealing...
  Advanced top doc [doc 2]: To prevent overfitting during neural network training, techn...
  Different? YES

────────────────────────────────────────────────────────────
Query: how does backpropagation assign credit to weights?
  Running Naïve RAG...
  Running Advanced RAG...
  Naïve top doc  [doc 1]: Backpropagation computes gradients layer-by-layer using the ...
  Advanced top doc [doc 1]: Backpr



### Observations
- **Query 1** : Both pipelines retrieve the same top document (Doc 3 on self-attention). This is expected — SBERT's semantic similarity is strong for well-known concepts. HyDE enriches the context ranking but the single best doc is the same.
- **Query 2** : The Advanced RAG pipeline retrieved Doc 7 (learning rate scheduling) as the top result after re-ranking, while Naïve RAG defaulted to Doc 6 (Adam optimizer). Both are valid, but the cross-encoder judged Doc 7 slightly more relevant given the broader phrasing "techniques" (plural). This demonstrates that hybrid retrieval + re-ranking can surface diverse, equally valid answers.
- **Query 3** : Both pipelines correctly identify Doc 1. The keyword "backpropagation" is strong enough that BM25 and SBERT agree, and the cross-encoder simply confirms this. Shows the pipeline is robust to already-specific queries.

In [22]:
for r in comparison_results:
    print(f"\n{'═'*70}")
    print(f"QUERY: {r['query']}")
    print(f"{'─'*70}")
    print(f"Naïve RAG  (doc {r['naive_top_doc_id']}): {r['naive_top_doc']}")
    print(f"   Answer: {r['naive_answer'][:300]}")
    print(f"\nAdvanced RAG (doc {r['adv_top_doc_id']}): {r['adv_top_doc']}")
    print(f"   Answer: {r['adv_answer'][:300]}")
    print(f"\n   Different top doc? {'YES' if r['are_different'] else 'No'}")


══════════════════════════════════════════════════════════════════════
QUERY: how do transformers encode meaning?
──────────────────────────────────────────────────────────────────────
Naïve RAG  (doc 3): The Transformer architecture relies on self-attention mechanisms to weigh the re...
   Answer: According to the provided context, the Transformer architecture encodes meaning through self-attention mechanisms, which weigh the relevance of every token against every other token in a sequence, enabling rich contextual representations.

Advanced RAG (doc 5): BERT (Bidirectional Encoder Representations from Transformers) pre-trains on mas...
   Answer: Transformers encode meaning through self-attention mechanisms, which weigh the relevance of every token against every other token in a sequence. This enables the model to capture rich contextual representations and understand the relationships between different tokens in the input sequence.

   Different top doc? YES

══════════════════════

---
## Bonus 1 — Weighted RRF


In [23]:
def weighted_rrf_retrieve(query: str, alpha: float, top_k: int = 5, k: int = 60) -> List[Dict]:

    bm25_ranking  = retriever._bm25_ranked(query)
    sbert_ranking = retriever._sbert_ranked(query)

    bm25_rank_map  = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranking)}
    sbert_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranking)}

    scores = {}
    for doc_id in range(len(corpus)):
        scores[doc_id] = (
            alpha       * (1.0 / (k + bm25_rank_map[doc_id]))  +
            (1 - alpha) * (1.0 / (k + sbert_rank_map[doc_id]))
        )

    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    results = []
    for doc_id, score in sorted_docs[:top_k]:
        results.append({
            "doc_id"            : doc_id,
            "weighted_rrf_score": round(score, 6),
            "bm25_rank"         : bm25_rank_map[doc_id],
            "sbert_rank"        : sbert_rank_map[doc_id],
            "text"              : corpus[doc_id],
        })
    return results


keyword_query  = "BLEU score brevity penalty"
semantic_query = "how does a neural network learn?"

print("Bonus 1 — Weighted RRF Experiment")
print("=" * 60)

for q_label, q in [("Keyword-heavy", keyword_query), ("Semantic", semantic_query)]:
    print(f"\n{q_label} query: '{q}'")
    print(f"{'alpha':>8}  {'top_doc_id':>12}  {'score':>10}  {'bm25_rank':>10}  {'sbert_rank':>11}")
    print("-" * 60)
    for alpha in [0.3, 0.5, 0.7]:
        results = weighted_rrf_retrieve(q, alpha=alpha, top_k=1)
        r = results[0]
        print(f"  α={alpha:.1f}   doc {r['doc_id']:>3}          "
              f"{r['weighted_rrf_score']:>10.6f}  "
              f"{r['bm25_rank']:>9}  "
              f"{r['sbert_rank']:>10}")
        print(f"          '{r['text'][:65]}...'")

Bonus 1 — Weighted RRF Experiment

Keyword-heavy query: 'BLEU score brevity penalty'
   alpha    top_doc_id       score   bm25_rank   sbert_rank
------------------------------------------------------------
  α=0.3   doc   9            0.016393          1           1
          'The BLEU score (Bilingual Evaluation Understudy) measures n-gram ...'
  α=0.5   doc   9            0.016393          1           1
          'The BLEU score (Bilingual Evaluation Understudy) measures n-gram ...'
  α=0.7   doc   9            0.016393          1           1
          'The BLEU score (Bilingual Evaluation Understudy) measures n-gram ...'

Semantic query: 'how does a neural network learn?'
   alpha    top_doc_id       score   bm25_rank   sbert_rank
------------------------------------------------------------
  α=0.3   doc   0            0.016314          2           1
          'Neural network training uses gradient descent to minimize a loss ...'
  α=0.5   doc   0            0.016261          2     

### Bonus 1 — Observations

- **Keyword-heavy query** (BLEU score brevity penalty): As α increases toward 0.7, BM25 gets more weight and correctly surfaces Doc 9 (the BLEU document with exact jargon). Dense SBERT embeddings struggle because "BLEU" and "brevity penalty" are rare terms.
- **Semantic query** : Lower α (e.g., 0.3) gives more weight to SBERT, which correctly maps the vague phrase to training-related documents. BM25 would miss the semantic intent.
- **Conclusion**: α = 0.7 is better for keyword/jargon queries; α = 0.3 is better for vague/semantic queries. α = 0.5 (standard RRF) is a good balanced default.

---
## Bonus 2 — Chunk Size Study

I decided to take a long document (>500 words), split it into chunks of 50, 100, and 200 words, and show how chunk size affects retrieval quality.

In [24]:
long_doc = """
The Transformer architecture, introduced by Vaswani et al. in the paper 'Attention Is All You Need',
revolutionized natural language processing by replacing recurrent networks with self-attention mechanisms.
Unlike RNNs and LSTMs that process tokens sequentially, the Transformer processes all tokens in parallel,
making it far more efficient on modern hardware. The core innovation is the multi-head self-attention
mechanism, which allows every token in a sequence to attend to every other token simultaneously.
Each attention head computes query, key, and value matrices from the input embeddings using learned
linear projections. The attention score between two tokens is the dot product of their query and key
vectors, scaled by the square root of the key dimensionality to stabilize gradients. A softmax
function converts these scores into a probability distribution, which is then used to compute a
weighted sum of the value vectors, producing the attended output for each position.
Multi-head attention stacks several such attention heads in parallel, each learning to attend to
different aspects of the sequence — one head might learn syntactic relationships while another
captures semantic similarity. The outputs of all heads are concatenated and projected back to the
model dimension. Positional encodings, either sinusoidal or learned, are added to the input
embeddings to inject sequence order information, since self-attention is permutation-invariant.
The encoder stack consists of alternating multi-head attention and feed-forward layers, each wrapped
in residual connections and layer normalisation. The feed-forward sublayer applies two linear
transformations with a ReLU or GELU activation in between, expanding the representation to a higher
dimension before projecting back down. The decoder adds a cross-attention layer that attends to
the encoder's output, enabling the model to condition generated tokens on the full input context.
The training objective for sequence-to-sequence tasks is typically teacher forcing with cross-entropy
loss over the target vocabulary at each position. Modern variants such as BERT use only the encoder
stack and are pre-trained with masked language modelling, while GPT uses only the decoder stack
with causal (autoregressive) masking. The scaling behaviour of Transformers, where performance
improves predictably with more parameters, data, and compute, has made them the dominant architecture
for large language models. Techniques such as rotary positional embeddings (RoPE), grouped-query
attention, and flash attention have been developed to extend context lengths and reduce memory
footprint, enabling models with hundreds of billions of parameters to be trained and deployed
efficiently on modern GPU clusters.
""".strip()

print(f"Long doc word count: {len(long_doc.split())} words")

Long doc word count: 396 words


In [26]:
def chunk_document(text: str, chunk_size: int) -> List[str]:
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk)
    return chunks

# Chunking study
chunk_query = "how does multi-head attention work in transformers?"

print(f"Chunk Size Study — Query: '{chunk_query}'")
print("=" * 70)

for size in [50, 100, 200]:
    chunks = chunk_document(long_doc, chunk_size=size)
    print(f"\nChunk size = {size} words  →  {len(chunks)} chunks produced")

    mini_retriever = HybridRetriever(corpus=chunks, k=60)
    results = mini_retriever.retrieve(chunk_query, top_k=1)

    if results:
        r = results[0]
        print(f"   Top chunk (doc_id={r['doc_id']}):")
        print(f"   '{r['text'][:200]}...'")
        print(f"   RRF={r['rrf_score']}  BM25_rank={r['bm25_rank']}  SBERT_rank={r['sbert_rank']}")

Chunk Size Study — Query: 'how does multi-head attention work in transformers?'

Chunk size = 50 words  →  8 chunks produced
BM25 index built.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT index built.
   Top chunk (doc_id=1):
   'The core innovation is the multi-head self-attention mechanism, which allows every token in a sequence to attend to every other token simultaneously. Each attention head computes query, key, and value...'
   RRF=0.032787  BM25_rank=1  SBERT_rank=1

Chunk size = 100 words  →  4 chunks produced
BM25 index built.


...

SBERT index built.
   Top chunk (doc_id=0):
   'The Transformer architecture, introduced by Vaswani et al. in the paper 'Attention Is All You Need', revolutionized natural language processing by replacing recurrent networks with self-attention mech...'
   RRF=0.032522  BM25_rank=1  SBERT_rank=2

Chunk size = 200 words  →  2 chunks produced
BM25 index built.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT index built.
   Top chunk (doc_id=0):
   'The Transformer architecture, introduced by Vaswani et al. in the paper 'Attention Is All You Need', revolutionized natural language processing by replacing recurrent networks with self-attention mech...'
   RRF=0.032522  BM25_rank=2  SBERT_rank=1


### Bonus 2 — Observations


**Conclusion:** Chunk size 100 words gave the best retrieval quality for this query — small enough to be focused, large enough to preserve necessary context.

---
## Bonus 3 — ColBERT as a Third Retriever in Hybrid RRF

ColBERT's MaxSim scoring: for each query token, finding its maximum similarity to any document token and then sum over all query tokens.

In [27]:
class ColBERTRetriever:
    """
    Implements ColBERT MaxSim scoring using SBERT token embeddings.
    For each query token embedding, find the maximum cosine similarity
    to any document token embedding. Sum these max similarities.
    """

    def __init__(self, corpus: List[str]):
        self.corpus = corpus
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")

        self.doc_token_embeddings = []
        for doc in corpus:
            tokens = doc.lower().split()
            embs = self.sbert.encode(tokens, convert_to_numpy=True, show_progress_bar=False)
            self.doc_token_embeddings.append(embs)  # shape: [num_tokens, dim]

        print(f"ColBERT token embeddings built for {len(corpus)} documents.")

    def maxsim_score(self, query: str, doc_idx: int) -> float:
        query_tokens = query.lower().split()
        q_embs = self.sbert.encode(query_tokens, convert_to_numpy=True, show_progress_bar=False)
        d_embs = self.doc_token_embeddings[doc_idx]

        # Normalise all embeddings for cosine similarity
        q_embs_norm = q_embs / (np.linalg.norm(q_embs, axis=1, keepdims=True) + 1e-9)
        d_embs_norm = d_embs / (np.linalg.norm(d_embs, axis=1, keepdims=True) + 1e-9)

        sim_matrix = q_embs_norm @ d_embs_norm.T

        # MaxSim: for each query token we take max over doc tokens and then sum
        maxsim = sim_matrix.max(axis=1).sum()
        return float(maxsim)

    def ranked(self, query: str) -> List[int]:
        scores = [self.maxsim_score(query, i) for i in range(len(self.corpus))]
        return np.argsort(scores)[::-1].tolist()

colbert_retriever = ColBERTRetriever(corpus=corpus)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ColBERT token embeddings built for 12 documents.


In [28]:
def three_way_hybrid_retrieve(query: str, top_k: int = 5, k: int = 60) -> List[Dict]:

    bm25_ranking    = retriever._bm25_ranked(query)
    sbert_ranking   = retriever._sbert_ranked(query)
    colbert_ranking = colbert_retriever.ranked(query)

    bm25_rank_map    = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranking)}
    sbert_rank_map   = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranking)}
    colbert_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(colbert_ranking)}

    rrf_scores = {}
    for doc_id in range(len(corpus)):
        rrf_scores[doc_id] = (
            (1.0 / (k + bm25_rank_map[doc_id]))    +
            (1.0 / (k + sbert_rank_map[doc_id]))   +
            (1.0 / (k + colbert_rank_map[doc_id]))
        )

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    results = []
    for doc_id, rrf_score in sorted_docs[:top_k]:
        results.append({
            "doc_id"        : doc_id,
            "rrf_score"     : round(rrf_score, 6),
            "bm25_rank"     : bm25_rank_map[doc_id],
            "sbert_rank"    : sbert_rank_map[doc_id],
            "colbert_rank"  : colbert_rank_map[doc_id],
            "text"          : corpus[doc_id],
        })
    return results


print("Three-way Hybrid (BM25 + SBERT + ColBERT) — 'how do transformers encode meaning?'")
results_3way = three_way_hybrid_retrieve("how do transformers encode meaning?", top_k=5)
print(f"{'doc_id':>8}  {'rrf_score':>12}  {'bm25_rank':>10}  {'sbert_rank':>11}  {'colbert_rank':>13}")
print("-" * 60)
for r in results_3way:
    print(f"  doc {r['doc_id']:>3}   {r['rrf_score']:>12.6f}  "
          f"{r['bm25_rank']:>10}  {r['sbert_rank']:>11}  {r['colbert_rank']:>13}")
    print(f"          → {r['text'][:70]}...")

Three-way Hybrid (BM25 + SBERT + ColBERT) — 'how do transformers encode meaning?'
  doc_id     rrf_score   bm25_rank   sbert_rank   colbert_rank
------------------------------------------------------------
  doc   4       0.048147           1            4              2
          → Attention in neural networks assigns a score to each word pair in a se...
  doc   5       0.047228           8            2              1
          → BERT (Bidirectional Encoder Representations from Transformers) pre-tra...
  doc   9       0.046883           3            5              4
          → The BLEU score (Bilingual Evaluation Understudy) measures n-gram overl...
  doc  10       0.046883           4            3              5
          → Convolutional Neural Networks (CNNs) use learnable filters that slide ...
  doc   3       0.046759           9            1              3
          → The Transformer architecture relies on self-attention mechanisms to we...
